# GPU Validation: KV Cache Tier Persistence (TinyLlama-1.1B)

Reruns the end-to-end TinyLlama cache save/restore validation on a GPU, producing
the GPU-side numbers for the paper's break-even analysis.

**Before running:** `Runtime` → `Change runtime type` → select **T4 GPU** → Save.
Then `Runtime` → `Run all` (~10–15 min). The last cell downloads
`tinyllama_gpu_results.json` — that file is the deliverable.

Every step asserts its own success: if any cell shows an error, the run is
invalid — copy the error text rather than the downloaded file.

In [ ]:
# Step 1: confirm a GPU is attached
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: set Runtime -> Change runtime type -> T4 GPU, then rerun"
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: clone the repo and install dependencies.
# transformers is PINNED to the version the integration script is validated
# against (Colab often ships a newer release with a changed cache API).
import os, sys
%cd /content
if not os.path.exists('kv-cache-tier-persistence'):
    !git clone -q https://github.com/aditya-lawankar/kv-cache-tier-persistence.git
%cd kv-cache-tier-persistence
!pip install -q -e . 'transformers==4.51.*' accelerate

# Verify the pin against what is ON DISK (importlib.metadata), not against a
# module object that may have been imported earlier in this kernel.
import importlib.metadata as _md
_ver = _md.version('transformers')
assert _ver.startswith('4.51'), f"transformers pin failed on disk: {_ver} — check the pip output above"

if 'transformers' in sys.modules and sys.modules['transformers'].__version__ != _ver:
    # A different transformers was imported earlier in this same runtime
    # (e.g. a previous failed attempt). Python cannot hot-swap it; restart.
    print('Pinned transformers is installed but a stale version is loaded in this kernel.')
    print('>>> The runtime will now restart (Colab shows a crash notice — EXPECTED). <<<')
    print('>>> When it settles, do Runtime -> Run all ONCE MORE to finish.          <<<')
    os.kill(os.getpid(), 9)

import torch, transformers, zstandard, lz4
print(f"transformers {transformers.__version__} | torch {torch.__version__}")

In [ ]:
# Step 3: run 1 — 256- and 512-token prompts, 3 trials each
!python benchmarks/tinyllama_integration.py --max-tokens 512 --trials 3 --output benchmarks/results/gpu_512
assert os.path.exists('benchmarks/results/gpu_512/tinyllama_integration_results.json'), \
    "Run 1 produced no results — scroll up in THIS cell for the traceback"

In [ ]:
# Step 4: run 2 — ~1850-token prompts (safely inside TinyLlama's 2048 context
# limit including 50 generated tokens; the prompt builder truncates precisely)
!python benchmarks/tinyllama_integration.py --max-tokens 1850 --trials 3 --output benchmarks/results/gpu_1850
assert os.path.exists('benchmarks/results/gpu_1850/tinyllama_integration_results.json'), \
    "Run 2 produced no results — scroll up in THIS cell for the traceback"

In [ ]:
# Step 5: merge, summarize, and download the deliverable
import json, glob

rows = []
for path in sorted(glob.glob('benchmarks/results/gpu_*/tinyllama_integration_results.json')):
    rows += json.load(open(path))
assert rows, "No results found — a run cell above failed; do not use the downloaded file"
rows.sort(key=lambda r: r['prompt_tokens'])

with open('tinyllama_gpu_results.json', 'w') as f:
    json.dump(rows, f, indent=2)

print(f"{'tokens':>7} {'cold TTFT':>10} {'warm TTFT':>10} {'TTFT x':>7} {'E2E x':>6} "
      f"{'save':>7} {'load':>7} {'restore':>8} {'match':>6}")
for r in rows:
    print(f"{r['prompt_tokens']:>7} {r['cold_ttft_ms']:>8.0f}ms {r['warm_ttft_ms']:>8.0f}ms "
          f"{r['ttft_speedup_x']:>6.2f}x {r['e2e_speedup_x']:>5.2f}x "
          f"{r['save_latency_ms']:>5.0f}ms {r['load_latency_ms']:>5.0f}ms "
          f"{r['restore_to_device_ms']:>6.0f}ms {'  yes' if r['semantic_match'] else '   NO'}")

from google.colab import files
files.download('tinyllama_gpu_results.json')